# Preparation HILDA+ data

In [ ]:
# Libraries
import os, time, shutil, rioxarray
import numpy as np
import xarray as xr
from scipy import ndimage
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir_data =  '../data/'
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'

---

### Pre-processing

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='compute', 
                       cores=24, #6
                       memory='256 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='01:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=10)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)

client

In [ ]:
# Function to trim memory of workers
def trim_memory() -> int:
    import ctypes
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

In [ ]:
def prep_hilda():
    """Prepare stable hilda grid cell data"""

    # Get hilda dataset, select years since 1960
    hilda = (
        xr.open_dataset(
            f"{dir_data}hilda/hildap_vGLOB-2.0-f_netcdf/hildaplus_GLOB-2-0_states.nc",
            decode_coords='all',
        )
        .chunk(dict(latitude=5000, longitude=5000, time=1))
        .LULC_states
        .sel(time=slice(1960, None))
    )

    # Aggregate to broad hilda classes:
    # 0:ocean, 1:urban, 2:crops, 3:pasture/rangeland, 4:forest,
    # 5:unmanaged grass/shrubland, 6:sparse/no vegetation, 7:water, 9:no data
    hilda_agg = (hilda.astype('int16') // 10).astype('int8')

    # Create dataset
    hilda2 = xr.Dataset(
        coords={
            'lat': hilda.latitude,
            'lon': hilda.longitude,
            'time': hilda.time.astype('int'),
        }
    )

    # Create one boolean (time, lat, lon) variable per class
    for i in [0, 1, 2, 3, 4, 5, 6, 7, 9]:
        hilda2[f'hilda_{i}'] = (hilda_agg == i)

    # Identify grid cells without land cover change (per variable)
    hilda_prep_sum = xr.Dataset(
        {v: (hilda2[v].sum('time') == hilda2.sizes['time']) for v in hilda2.data_vars}
    )

    # Store arrays without landcover change in dataset
    hilda_prep_stable = xr.Dataset()
    hilda_prep_stable['hilda_stable_ocean'] = hilda_prep_sum['hilda_0']
    hilda_prep_stable['hilda_stable_urban'] = hilda_prep_sum['hilda_1']
    hilda_prep_stable['hilda_stable_crop'] = hilda_prep_sum['hilda_2']
    hilda_prep_stable['hilda_stable_pasture'] = hilda_prep_sum['hilda_3']
    hilda_prep_stable['hilda_stable_forest'] = hilda_prep_sum['hilda_4']
    hilda_prep_stable['hilda_stable_shrub'] = hilda_prep_sum['hilda_5']
    hilda_prep_stable['hilda_stable_other'] = hilda_prep_sum['hilda_6']
    hilda_prep_stable['hilda_stable_water'] = hilda_prep_sum['hilda_7']
    hilda_prep_stable['hilda_stable_nodata'] = hilda_prep_sum['hilda_9']

    hilda_prep_stable = hilda_prep_stable.persist()

    # Change dtype for regridding, export as tif
    for v in hilda_prep_stable.data_vars:
        (
            hilda_prep_stable[v]
            .rename(dict(lat='y', lon='x'))
            .astype('float32')
            .rio.to_raster(f'{dir01}{v}.tif')
        )

%time prep_hilda()

In [ ]:
# Close dask cluster
cluster.close()

In [ ]:
# Wait for dask client to completely disconnect
time.sleep(60)

---

### Regridding

In [ ]:
# Import regridding function
from regrid_high_res_v1_01 import regrid_high_res, prep_tif

In [ ]:
def regrid_da(f_source, dir_target, dir_source, dir_out, 
              size_tiles, fill_value=None, olap=1):  
    """Regrid large xarray dataarrays.

    Args:
        f_source (str): The filename (without extension) of the source .tif file to be regridded.
        dir_target (str): Directory containing target grid .tif file.
        dir_source (str): Directory containing the the source  .tif file.
        dir_out (str): Directory to store the output and intermediate files.
        size_tiles (int): Size of the regridding tiles in degrees.
        fill_value (float, optional): Fill value to use in the regridding process. Defaults to None.
        olap (int, optional): Overlap size in degrees for regridding tiles. Defaults to 1.
        
    Returns:
        xarray.Dataset: The combined dataset after regridding.
    """
    # Prepare the target and source data arrays from TIFF files
    da_target = prep_tif(dir_target + 'target_grid_r001.tif', 'target_grid_r001')
    da_source = prep_tif(dir_source + f_source + '.tif', f_source)
    # Regridd source array to target grid
    regrid_high_res(da_target, da_source, dir_out,
                    account='bm0891', partition='shared',
                    size_tiles=size_tiles, olap=olap, fill_value = fill_value,
                    type_export='zarr', del_interm=False)

In [ ]:
# Regridding
for i in ['hilda_stable_ocean', 'hilda_stable_urban', 'hilda_stable_crop', 
          'hilda_stable_pasture', 'hilda_stable_forest', 'hilda_stable_shrub', 
          'hilda_stable_other', 'hilda_stable_water', 'hilda_stable_nodata']:
    %time regrid_da(i, dir01, dir01, dir01, 30, np.nan, 0.1)
# Takes about 4min for one file

---

### Fill land nans

In [ ]:
def fill_nans(var, dir_out):
    """
    Fills NaN values in the specified variable's dataset using the nearest valid 
    data, applies a land mask, and exports the result to a new Zarr dataset.

    Args:
        var (str): Name of the variable to process (e.g., 'temperature', 'precipitation').
        dir_out (str): Directory where prepared data is stored and the filled dataset will be exported.

    Returns:
        None
    """

    def fill_nans_array(data, invalid):
        """
        Replace invalid (NaN) data cells by the value of the nearest valid data 
        cell.
        """
        ind = ndimage.distance_transform_edt(invalid,
                                             return_distances=False,
                                             return_indices=True)
        return data[tuple(ind)]

    # Paths for input and output
    land_mask_path = os.path.join(dir_out, 'ds_prep_copernicus_land_mask_r001.zarr')
    var_data_path = os.path.join(dir_out, f'ds_regridded_{var}.zarr')
    output_path = os.path.join(dir_out, f'ds_prep_{var}_r001.zarr')

    # Read land mask data
    da_land = xr.open_zarr(land_mask_path) \
                .chunk(dict(lat=5000, lon=5000)) \
                .copernicus_land_mask \
                .compute()

    # Read variable data
    da_var = xr.open_zarr(var_data_path)['regridded_' + var]

    # Fill nan using function fill_nans_array
    # If there are no NaNs, skip filling process
    if not da_var.isnull().any():
        da_fill = da_var.values  # No filling required
    else:
        da_fill = fill_nans_array(da_var.values, da_var.isnull().values)

    # Create a new Dataset with filled data
    ds_filled = xr.Dataset(dict(lat = da_var.lat, lon=da_var.lon))
    ds_filled[var] = (('lat', 'lon'), da_fill)

    # Apply land mask to the filled data
    ds_filled = ds_filled.where(da_land)

    # Export the filled dataset to Zarr format
    ds_filled.chunk(dict(lat=5000, lon=5000)) \
             .to_zarr(output_path, mode='w')


for i in ['hilda_stable_ocean', 'hilda_stable_urban', 'hilda_stable_crop', 
          'hilda_stable_pasture', 'hilda_stable_forest', 'hilda_stable_shrub', 
          'hilda_stable_other', 'hilda_stable_water', 'hilda_stable_nodata']:
    %time fill_nans(i, dir01)

---

### Post-processing

In [ ]:
xr.open_mfdataset(f'{dir01}ds_prep_hilda_stable_*_r001.zarr', 
                  engine='zarr') \
    .to_zarr(f'{dir01}ds_prep_hilda_stable_r001.zarr', mode='w');

In [ ]:
for i in ['hilda_stable_ocean', 'hilda_stable_urban', 'hilda_stable_crop', 
          'hilda_stable_pasture', 'hilda_stable_forest', 'hilda_stable_shrub', 
          'hilda_stable_other', 'hilda_stable_water', 'hilda_stable_nodata']:
    shutil.rmtree(f'{dir01}ds_prep_{i}_r001.zarr')